# Hybrid recommendation

The assumed task is added to the observed baseline. Savings are scenario outcomes. Allowed slots are enforced before ranking.

Run the complete pipeline first. Select the project virtual environment as the notebook kernel.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [2]:
from src.energy import simulated_tariff, hybrid_scores, simulate_day
import joblib
model = joblib.load(ROOT/'results/models/ImplicitALS_11.joblib')['model']
pref = model.predict('H0000',np.arange(48))
train = pd.read_csv(ROOT/'data/processed/train.csv')
peak = train.groupby('slot').kwh.mean().reindex(range(48)).to_numpy()
score, parts = hybrid_scores(pref,simulated_tariff(),peak,36,8,[.4,.3,.2,.1])
feasible = np.arange(28,45)
best = int(feasible[np.argmax(score[feasible])])
print('Illustrative candidate:',f'{best//2:02d}:{30*(best%2):02d}')

Illustrative candidate: 21:00


In [3]:
recommendations = pd.read_csv(ROOT/'results/tables/recommendations.csv')
recommendations.query("household_id == 'H0000' and seed == 11 and daytype == 0")

Out[0]: 
   seed  ...                                        explanation
0    11  ...  Recommend 21:00: tariff £0.24/kWh vs £0.40 at ...
1    11  ...  Recommend 21:30: tariff £0.24/kWh vs £0.40 at ...
2    11  ...  Recommend 20:30: tariff £0.24/kWh vs £0.40 at ...
3    11  ...  Recommend 20:00: tariff £0.24/kWh vs £0.40 at ...
4    11  ...  Recommend 16:30: tariff £0.40/kWh vs £0.40 at ...

[5 rows x 21 columns]


In [4]:
pd.read_csv(ROOT/'results/tables/energy.csv').groupby('strategy')[['savings','savings_percent','inconvenience']].mean()

Out[0]: 
                  savings  savings_percent  inconvenience
strategy                                                 
cost_only        0.160000         5.209265       1.000000
hybrid           0.160000         5.209265       0.783179
no_shift         0.000000         0.000000       0.000000
preference_only  0.043951         1.499839       0.364969
